# **Main Backend Code**

In [3]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')
# 1. CACHE & TRAIN THE BALANCED BASELINE MODEL
if 'baseline_model' not in globals() or not hasattr(baseline_model, 'classes_'):
    print("Training BALANCED Logistic Regression baseline model... (Takes ~10 seconds)")
    try:
        # Try loading the new dataset name, fallback to the original if it fails
        df = pd.read_csv('final_dataset (1)_2.csv')
    except FileNotFoundError:
        df = pd.read_csv('final_dataset (1).csv')

    X = df.drop('diabetes', axis=1)
    y = df['diabetes']

    # Using Logistic Regression as the baseline with class_weight='balanced'
    baseline_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
    baseline_model.fit(X, y)
    print("✅ Balanced Logistic Regression model trained and cached!\n")
else:
    print("⚡ Using cached balanced model\n")

def get_input(prompt, default):
    """Helper function to allow pressing 'Enter' to use a default value"""
    val = input(f"{prompt} [Default: {default}]: ")
    if val.strip() == '':
        return default
    return float(val)

# 2. CONTINUOUS LOOP
while True:
    print("-" * 55)
    print("Enter patient details (Press ENTER to use healthy defaults, type 'q' to quit):")

    val = input("1. High Blood Pressure (0=No, 1=Yes) [or 'q' to quit]: ")
    if val.lower() == 'q':
        print("Exiting predictor.")
        break

    try:
        # Default healthy baseline values
        high_bp = float(val) if val.strip() != '' else 0.0
        high_chol = get_input("2. High Cholesterol (0=No, 1=Yes)", 0.0)
        chol_check = get_input("3. Cholesterol Check in 5 yrs (0=No, 1=Yes)", 1.0)
        bmi = get_input("4. BMI (e.g., 25.0)", 25.0)
        smoker = get_input("5. Smoker (100+ cigarettes in life) (0=No, 1=Yes)", 0.0)
        stroke = get_input("6. Stroke history (0=No, 1=Yes)", 0.0)
        heart_disease = get_input("7. Heart disease/attack history (0=No, 1=Yes)", 0.0)
        phys_activity = get_input("8. Physical Activity in past 30 days (0=No, 1=Yes)", 1.0)
        fruits = get_input("9. Consume fruits daily (0=No, 1=Yes)", 1.0)
        veggies = get_input("10. Consume veggies daily (0=No, 1=Yes)", 1.0)
        heavy_alcohol = get_input("11. Heavy Alcohol consumer (0=No, 1=Yes)", 0.0)
        healthcare = get_input("12. Has Healthcare coverage (0=No, 1=Yes)", 1.0)
        no_doc_cost = get_input("13. Avoided doctor due to cost (0=No, 1=Yes)", 0.0)
        gen_hlth = get_input("14. General Health (1=Excellent to 5=Poor)", 2.0)
        ment_hlth = get_input("15. Days of poor mental health (0-30)", 0.0)
        phys_hlth = get_input("16. Days of poor physical health (0-30)", 0.0)
        diff_walk = get_input("17. Difficulty walking/climbing (0=No, 1=Yes)", 0.0)
        sex = get_input("18. Sex (0=Female, 1=Male)", 0.0)
        age = get_input("19. Age Category (1=18-24, 7=45-49, 13=80+)", 5.0)
        education = get_input("20. Education Level (1=No school, 6=College Grad)", 5.0)
        income = get_input("21. Income Level (1=<$10k, 8=>=$75k)", 6.0)

        # Must map inputs in the exact column order of the DataFrame
        input_data = [[
            high_bp, high_chol, chol_check, bmi, smoker, stroke, heart_disease,
            phys_activity, fruits, veggies, heavy_alcohol, healthcare, no_doc_cost,
            gen_hlth, ment_hlth, phys_hlth, diff_walk, sex, age, education, income
        ]]

        # Extract prediction probability
        risk_prob = baseline_model.predict_proba(input_data)[0][1]

        # Output Results
        print("\n" + "="*40)
        print("         PREDICTION RESULT")
        print("="*40)
        if risk_prob > 0.5:
            print(f"⚠️  HIGH RISK (Probability: {risk_prob:.1%})")
        else:
            print(f"✅  LOW RISK (Probability: {risk_prob:.1%})")
        print("="*40 + "\n")

    except ValueError:
        print("\n❌ Invalid input. Let's start this patient profile over.\n")

Training BALANCED Logistic Regression baseline model... (Takes ~10 seconds)
✅ Balanced Logistic Regression model trained and cached!

-------------------------------------------------------
Enter patient details (Press ENTER to use healthy defaults, type 'q' to quit):


KeyboardInterrupt: Interrupted by user

# **Model Testing & Evaluation — Performance Metrics, Cross-Validation and Comparison**

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import warnings

warnings.filterwarnings('ignore')

print("Loading dataset and preparing cross-validation...\n")
df = pd.read_csv('/content/final_dataset (1).csv')
X = df.drop('diabetes', axis=1)
y = df['diabetes']

# Define the models to compare (all balanced to handle the class imbalance)
models = {
    "Logistic Regression (Baseline)": LogisticRegression(class_weight='balanced', max_iter=500, random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', max_depth=10, random_state=42),
    "Random Forest (Proposed)": RandomForestClassifier(n_estimators=50, max_depth=10, class_weight='balanced', random_state=42)
}

# Define the evaluation metrics
scoring = ['accuracy', 'precision', 'recall', 'f1']

# Execute 5-fold cross-validation for each model
for name, model in models.items():
    print(f"Evaluating {name}...")
    # cv=5 splits the data into 5 chunks, training on 4 and testing on 1, repeating 5 times
    scores = cross_validate(model, X, y, cv=5, scoring=scoring, n_jobs=-1)

    print(f"  Mean Accuracy:  {scores['test_accuracy'].mean():.4f}")
    print(f"  Mean Precision: {scores['test_precision'].mean():.4f}")
    print(f"  Mean Recall:    {scores['test_recall'].mean():.4f}")
    print(f"  Mean F1-Score:  {scores['test_f1'].mean():.4f}")
    print("-" * 40)

Loading dataset and preparing cross-validation...

Evaluating Logistic Regression (Baseline)...
  Mean Accuracy:  0.7312
  Mean Precision: 0.3115
  Mean Recall:    0.7652
  Mean F1-Score:  0.4426
----------------------------------------
Evaluating Decision Tree...
  Mean Accuracy:  0.7087
  Mean Precision: 0.2950
  Mean Recall:    0.7815
  Mean F1-Score:  0.4280
----------------------------------------
Evaluating Random Forest (Proposed)...
  Mean Accuracy:  0.7313
  Mean Precision: 0.3124
  Mean Recall:    0.7713
  Mean F1-Score:  0.4446
----------------------------------------


# ***Front-End Code***

In [ ]:
pip install fastapi uvicorn pandas scikit-learn pydantic

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import warnings

warnings.filterwarnings('ignore')

app = FastAPI(title="Diabetes Risk Predictor API")

# Enable CORS so your React frontend can communicate with this backend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], # In production, replace "*" with your React app's URL (e.g., "http://localhost:5173")
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Global variable to hold our model
rf_model_full = None

# Train or load the model when the server starts
@app.on_event("startup")
def load_model():
    global rf_model_full
    print("Training BALANCED model on ALL features... (Takes ~10 seconds)")
    try:
        df = pd.read_csv('final_dataset (1)_2.csv')
    except FileNotFoundError:
        try:
            df = pd.read_csv('final_dataset (1).csv')
        except FileNotFoundError:
            print("❌ Dataset not found! Please ensure the CSV is in the same directory.")
            return

    X = df.drop('diabetes', axis=1)
    y = df['diabetes']

    rf_model_full = RandomForestClassifier(
        n_estimators=50,
        max_depth=10,
        random_state=42,
        class_weight='balanced'
    )
    rf_model_full.fit(X, y)
    print("✅ Balanced 21-feature model trained and ready to serve!")

# Define the expected data structure from the React frontend
class PatientData(BaseModel):
    high_bp: float
    high_chol: float
    chol_check: float
    bmi: float
    smoker: float
    stroke: float
    heart_disease: float
    phys_activity: float
    fruits: float
    veggies: float
    heavy_alcohol: float
    healthcare: float
    no_doc_cost: float
    gen_hlth: float
    ment_hlth: float
    phys_hlth: float
    diff_walk: float
    sex: float
    age: float
    education: float
    income: float

@app.post("/predict")
def predict_risk(data: PatientData):
    if rf_model_full is None:
        raise HTTPException(status_code=500, detail="Model is not loaded.")

    # Map inputs in the exact column order expected by the model
    input_data = [[
        data.high_bp, data.high_chol, data.chol_check, data.bmi, data.smoker,
        data.stroke, data.heart_disease, data.phys_activity, data.fruits,
        data.veggies, data.heavy_alcohol, data.healthcare, data.no_doc_cost,
        data.gen_hlth, data.ment_hlth, data.phys_hlth, data.diff_walk,
        data.sex, data.age, data.education, data.income
    ]]

    # Extract prediction probability
    risk_prob = rf_model_full.predict_proba(input_data)[0][1]

    # Return as JSON to the frontend
    return {
        "risk_prob": float(risk_prob),
        "is_high_risk": bool(risk_prob > 0.5)
    }

In [ ]:
!ls

'final_dataset (1).csv'   sample_data


In [ ]:
code = """
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import warnings

warnings.filterwarnings('ignore')

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

rf_model_full = None

@app.on_event("startup")
def load_model():
    global rf_model_full
    try:
        df = pd.read_csv('final_dataset (1)_2.csv')
    except FileNotFoundError:
        df = pd.read_csv('final_dataset (1).csv')

    X = df.drop('diabetes', axis=1)
    y = df['diabetes']
    rf_model_full = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, class_weight='balanced')
    rf_model_full.fit(X, y)

class PatientData(BaseModel):
    high_bp: float
    high_chol: float
    chol_check: float
    bmi: float
    smoker: float
    stroke: float
    heart_disease: float
    phys_activity: float
    fruits: float
    veggies: float
    heavy_alcohol: float
    healthcare: float
    no_doc_cost: float
    gen_hlth: float
    ment_hlth: float
    phys_hlth: float
    diff_walk: float
    sex: float
    age: float
    education: float
    income: float

@app.post("/predict")
def predict_risk(data: PatientData):
    input_data = [[
        data.high_bp, data.high_chol, data.chol_check, data.bmi, data.smoker,
        data.stroke, data.heart_disease, data.phys_activity, data.fruits,
        data.veggies, data.heavy_alcohol, data.healthcare, data.no_doc_cost,
        data.gen_hlth, data.ment_hlth, data.phys_hlth, data.diff_walk,
        data.sex, data.age, data.education, data.income
    ]]
    risk_prob = rf_model_full.predict_proba(input_data)[0][1]
    return {"risk_prob": float(risk_prob), "is_high_risk": bool(risk_prob > 0.5)}
"""

with open("main.py", "w") as file:
    file.write(code)

print("✅ main.py has been successfully created!")

✅ main.py has been successfully created!


In [ ]:
!cat main.py


from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import warnings

warnings.filterwarnings('ignore')

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

rf_model_full = None

@app.on_event("startup")
def load_model():
    global rf_model_full
    try:
        df = pd.read_csv('final_dataset (1)_2.csv')
    except FileNotFoundError:
        df = pd.read_csv('final_dataset (1).csv')

    X = df.drop('diabetes', axis=1)
    y = df['diabetes']
    rf_model_full = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, class_weight='balanced')
    rf_model_full.fit(X, y)

class PatientData(BaseModel):
    high_bp: float
    high_chol: float
    chol_check: float
    bmi: float
    smoker: float
    stroke: float
    heart_disease: floa

In [ ]:
!uvicorn main:app --reload

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [38091] using WatchFiles
INFO:     Started server process [38093]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [38093]
INFO:     Stopping reloader process [38091]


In [ ]:
!npm install -g localtunnel
!uvicorn main:app --host 0.0.0.0 --port 8000 & lt --port 8000

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 1s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸your url is: https://afraid-walls-shop.loca.lt
INFO:     Started server process [38588]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     122.161.73.152:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     122.161.73.152:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     66.249.74.2:0 - "GET /robots.txt HTTP/1.1" 404 Not Found
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [38588]
^C


In [ ]:
!pip install gradio

In [5]:
import gradio as gr
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import warnings

warnings.filterwarnings('ignore')

# 1. Train the model
print("Training BALANCED model on ALL features...")
try:
    df = pd.read_csv('final_dataset (1)_2.csv')
except FileNotFoundError:
    df = pd.read_csv('final_dataset (1).csv')

X = df.drop('diabetes', axis=1)
y = df['diabetes']
rf_model_full = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, class_weight='balanced')
rf_model_full.fit(X, y)
print("✅ Model ready!")

# 2. Prediction logic
def predict_diabetes(high_bp, high_chol, chol_check, bmi, smoker, stroke, heart_disease,
                     phys_activity, fruits, veggies, heavy_alcohol, healthcare, no_doc_cost,
                     gen_hlth, ment_hlth, phys_hlth, diff_walk, sex, age, education, income):

    yes_no = lambda val: 1.0 if val == "Yes" else 0.0

    input_data = [[
        yes_no(high_bp), yes_no(high_chol), yes_no(chol_check), float(bmi),
        yes_no(smoker), yes_no(stroke), yes_no(heart_disease),
        yes_no(phys_activity), yes_no(fruits), yes_no(veggies),
        yes_no(heavy_alcohol), yes_no(healthcare), yes_no(no_doc_cost),
        float(gen_hlth), float(ment_hlth), float(phys_hlth), yes_no(diff_walk),
        1.0 if sex == "Male" else 0.0, float(age), float(education), float(income)
    ]]

    risk_prob = rf_model_full.predict_proba(input_data)[0][1]

    if risk_prob > 0.5:
        return f"""
        <div style='background: linear-gradient(to right, #fee2e2, #fecaca); padding: 30px; border-radius: 15px; border-left: 10px solid #ef4444; color: #7f1d1d; text-align: center; box-shadow: 0 10px 15px -3px rgba(0,0,0,0.1);'>
            <h2 style='margin: 0; font-size: 36px; font-weight: 900; letter-spacing: 1px;'>⚠️ HIGH RISK DETECTED</h2>
            <p style='margin: 15px 0 0 0; font-size: 22px;'>Estimated Probability: <b>{risk_prob:.1%}</b></p>
        </div>
        """
    else:
        return f"""
        <div style='background: linear-gradient(to right, #dcfce3, #bbf7d0); padding: 30px; border-radius: 15px; border-left: 10px solid #22c55e; color: #14532d; text-align: center; box-shadow: 0 10px 15px -3px rgba(0,0,0,0.1);'>
            <h2 style='margin: 0; font-size: 36px; font-weight: 900; letter-spacing: 1px;'>✅ LOW RISK</h2>
            <p style='margin: 15px 0 0 0; font-size: 22px;'>Estimated Probability: <b>{risk_prob:.1%}</b></p>
        </div>
        """

# 3. Custom CSS
css = """
.custom-header { text-align: center; padding: 30px; background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%); color: white; border-radius: 15px; margin-bottom: 25px; box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.1); }
.custom-header h1 { margin: 0; font-size: 40px; font-weight: 800; color: white !important; }
.custom-header p { margin: 10px 0 0 0; font-size: 18px; opacity: 0.9; }
.custom-header a { color: #bfdbfe; text-decoration: none; font-weight: bold; }
.custom-header a:hover { text-decoration: underline; color: white; }
.section-title { font-size: 1.3rem !important; font-weight: 700 !important; color: white !important; padding-bottom: 8px; border-bottom: 2px solid #475569; margin-bottom: 15px !important;}
.custom-footer { text-align: center; padding: 20px; margin-top: 30px; font-size: 15px; color: #94a3b8; border-top: 1px solid #334155; }
.custom-footer a { color: #60a5fa; text-decoration: none; font-weight: 600; }
.custom-footer a:hover { text-decoration: underline; }
"""

# 4. Build the UI
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate"), css=css, title="Diabetes Risk Predictor") as app:

    # Header with Contact Info
    gr.HTML('''
    <div class="custom-header">
        <h1>🩺 Advanced Patient Dashboard</h1>
        <p>AI-Powered Diabetes Risk Assessment</p>
        <p style="font-size: 14px; margin-top: 5px;">Contact: <a href="mailto:aryansrivastava737@gmail.com">aryansrivastava737@gmail.com</a></p>
    </div>
    ''')

    with gr.Row():
        with gr.Column():
            with gr.Group():
                gr.Markdown("<div class='section-title'>📊 Vitals & Health Scores</div>")
                bmi = gr.Number(label="BMI (Body Mass Index)", value=25.0)
                gen_hlth = gr.Slider(minimum=1, maximum=5, step=1, label="General Health Score (1=Excellent to 5=Poor)", value=2)
                phys_hlth = gr.Slider(minimum=0, maximum=30, step=1, label="Days of Poor Physical Health (past 30 days)", value=0)
                ment_hlth = gr.Slider(minimum=0, maximum=30, step=1, label="Days of Poor Mental Health (past 30 days)", value=0)

            with gr.Group():
                gr.Markdown("<div class='section-title'>👤 Demographics & Access</div>")
                sex = gr.Radio(choices=["Female", "Male"], label="Sex", value="Female")
                age = gr.Slider(minimum=1, maximum=13, step=1, label="Age Category (1=18-24 ... 13=80+)", value=5)
                education = gr.Slider(minimum=1, maximum=6, step=1, label="Education Level (1=No school ... 6=College Grad)", value=5)
                income = gr.Slider(minimum=1, maximum=8, step=1, label="Income Level (1=<$10k ... 8=>=$75k)", value=6)
                healthcare = gr.Radio(choices=["No", "Yes"], label="Has Healthcare Coverage", value="Yes")
                no_doc_cost = gr.Radio(choices=["No", "Yes"], label="Avoided Doctor Due to Cost", value="No")

        with gr.Column():
            with gr.Group():
                gr.Markdown("<div class='section-title'>🏥 Medical Conditions</div>")
                high_bp = gr.Radio(choices=["No", "Yes"], label="High Blood Pressure", value="No")
                high_chol = gr.Radio(choices=["No", "Yes"], label="High Cholesterol", value="No")
                chol_check = gr.Radio(choices=["No", "Yes"], label="Cholesterol Checked (Last 5 yrs)", value="Yes")
                stroke = gr.Radio(choices=["No", "Yes"], label="History of Stroke", value="No")
                heart_disease = gr.Radio(choices=["No", "Yes"], label="History of Heart Disease", value="No")
                diff_walk = gr.Radio(choices=["No", "Yes"], label="Difficulty Walking or Climbing Stairs", value="No")

            with gr.Group():
                gr.Markdown("<div class='section-title'>🏃 Lifestyle & Habits</div>")
                smoker = gr.Radio(choices=["No", "Yes"], label="Smoker (100+ lifetime cigarettes)", value="No")
                phys_activity = gr.Radio(choices=["No", "Yes"], label="Physical Activity (past 30 days)", value="Yes")
                fruits = gr.Radio(choices=["No", "Yes"], label="Consume Fruits Daily", value="Yes")
                veggies = gr.Radio(choices=["No", "Yes"], label="Consume Veggies Daily", value="Yes")
                heavy_alcohol = gr.Radio(choices=["No", "Yes"], label="Heavy Alcohol Consumer", value="No")

    # Action Area
    gr.Markdown("<br>")
    submit_btn = gr.Button("🔍 Analyze Patient Data & Calculate Risk", variant="primary", size="lg")

    with gr.Row():
        result_html = gr.HTML()

    input_components = [
        high_bp, high_chol, chol_check, bmi, smoker, stroke, heart_disease,
        phys_activity, fruits, veggies, heavy_alcohol, healthcare, no_doc_cost,
        gen_hlth, ment_hlth, phys_hlth, diff_walk, sex, age, education, income
    ]

    submit_btn.click(fn=predict_diabetes, inputs=input_components, outputs=result_html)

    # Contact Footer
    gr.HTML('''
    <div class="custom-footer">
        For support, inquiries, or feedback, please contact: <a href="mailto:aryansrivastava737@gmail.com">aryansrivastava737@gmail.com</a>
    </div>
    ''')

# Launch
app.launch(share=True)

Training BALANCED model on ALL features...
✅ Model ready!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://10ef868736a350e72e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
from google.colab import auth
auth.authenticate_user()

In [7]:
import os

# ... [your existing Interactive Diabetes Predictor code] ...

# Retrieve the port from Cloud Run's environment variables (defaults to 8080)
port = int(os.environ.get("PORT", 8080))

# Launch the interface with the required network configurations
demo.launch(server_name="0.0.0.0", server_port=port)

NameError: name 'demo' is not defined